In [ ]:

!git clone https://github.com/viveknarayana/ecs174-dior.git /content/ecs174-dior
%cd /content/ecs174-dior

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Download and extract the four public ZIPs. This transfers about 7 GB; run when ready.
!pip -q install gdown
!mkdir -p /content/dior_archives /content/DIOR
!gdown --folder 'https://drive.google.com/drive/folders/1UdlgHk49iu6WpcJ5467iT-UqNPpx__CC' -O /content/dior_archives

from pathlib import Path
from zipfile import ZipFile
archive_dir = Path('/content/dior_archives')
data_root = Path('/content/DIOR')
for archive in sorted(archive_dir.glob('*.zip')):
    print('Extracting', archive.name)
    with ZipFile(archive) as zf:
        zf.extractall(data_root)
print('Extracted:', sorted(p.name for p in data_root.iterdir()))

In [ ]:
# Pull the latest project updates before using the cloned source.
!git -C /content/ecs174-dior pull origin main

In [ ]:
# Train one epoch per execution, show live progress, and download its outputs.
import importlib
import csv
import subprocess
import shutil
import sys
from pathlib import Path
import torch
import config
from google.colab import files
CONFIG = importlib.reload(config).CONFIG
OUTPUT_ROOT = '/content/dior_outputs'

def run_one_epoch(backbone='resnet50', resume=None, tiny=False):
    run_dir = Path(OUTPUT_ROOT) / backbone
    checkpoint = Path(resume) if resume else None
    if checkpoint is None and list(run_dir.glob('checkpoint_epoch_*.pth')):
        raise ValueError('Existing checkpoints found. Pass a checkpoint to resume or choose a fresh output folder.')
    previous_epoch = 0
    if checkpoint:
        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
        previous_epoch = int(saved['epoch'])
        log_path = run_dir / 'training_log.csv'
        uploaded_log = Path('/content/training_log.csv')
        if not log_path.is_file() and uploaded_log.is_file():
            run_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(uploaded_log, log_path)
        if not log_path.is_file():
            raise ValueError('Upload the matching training_log.csv to the Colab session first.')
        with log_path.open(newline='') as f:
            rows = list(csv.DictReader(f))
        if not rows or int(rows[-1]['epoch']) != previous_epoch:
            raise ValueError('The CSV log must end at the checkpoint epoch.')
        print(f'Resuming from epoch {previous_epoch}.', flush=True)
        del saved
    next_epoch = previous_epoch + 1
    if next_epoch > 10:
        raise ValueError('The 10-epoch target is already complete.')
    command = [sys.executable, '-u', 'train.py', '--data-root', '/content/DIOR',
               '--backbone', backbone, '--output-root', OUTPUT_ROOT,
               '--epochs', str(next_epoch), '--batch-size', '2']
    if checkpoint:
        command += ['--resume', str(checkpoint)]
    if tiny:
        command += ['--tiny-test', '--tiny-train-images', '2', '--tiny-validation-images', '1']
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)
    print(f'Epoch {next_epoch} complete. Downloading checkpoint, CSV, and plot.', flush=True)
    for name in (f'checkpoint_epoch_{next_epoch:02d}.pth', 'training_log.csv', 'loss_curves.png'):
        path = run_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        files.download(str(path))


In [ ]:
# One epoch on two training images and one validation image.
run_training('resnet50', ['--tiny-test', '--tiny-train-images', '2', '--tiny-validation-images', '1', '--epochs', '1'], output_root='/content/smoke_test_outputs')

In [ ]:
# Resume epoch 1 checkpoint, train epoch 2, then download its outputs.
run_one_epoch('resnet50', resume='/content/checkpoint_epoch_01.pth')


In [ ]:
# Full MobileNetV3 run.
run_training('mobilenet')

In [ ]:
# Load the smoke-test checkpoint and inspect predictions on one training image.
import sys
sys.path.insert(0, '/content/ecs174-dior/DIOR_Data')
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from dataset import DIORDataset
from train import build_model
from classes import CLASS_NAMES

checkpoint_path = '/content/smoke_test_outputs/resnet50/downloads/resnet50_smoke_size640_batch2_lr0.005_epochs1_20261116_231410_epoch01_checkpoint_epoch_01.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model = build_model(checkpoint['backbone'], checkpoint['config']['image_size'])
model.load_state_dict(checkpoint['model_state_dict'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()
dataset = DIORDataset('/content/DIOR', split='train')
image, target = dataset[0]
with torch.inference_mode():
    prediction = model([image.to(device)])[0]
print('Checkpoint epoch:', checkpoint['epoch'], 'Image ID:', dataset.ids[0])
print('Top 5 predicted classes and confidence scores:')
for score, label in zip(prediction['scores'][:5].cpu(), prediction['labels'][:5].cpu()):
    print(f'{CLASS_NAMES[label.item()-1]}: {score.item():.3f}')
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image.permute(1, 2, 0))
for box, score, label in zip(prediction['boxes'].cpu(), prediction['scores'].cpu(), prediction['labels'].cpu()):
    if score < 0.25:
        continue
    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='red', linewidth=2))
    ax.text(x1, y1, f'{CLASS_NAMES[label.item()-1]} {score:.2f}', color='white', backgroundcolor='red')
ax.set_title('Smoke-test predictions, confidence ≥ 0.25')
ax.axis('off')
plt.show()